# SmartEye — Entrenamiento y evaluación en Google Colab

Esta notebook es el **punto de entrada de Colab** para el proyecto SmartEye.

- El código reutilizable está en `src/`.
- Los datos de entrenamiento se preparan en `/content` para aprovechar el disco local de Colab.
- Modelos, métricas, gráficos y logs se guardan en Google Drive para que no se pierdan al reiniciar el runtime.
- El flujo implementado corresponde al experimento **ResNet50 de clasificación de imágenes** imágenes en escala de grises, conversión a RGB, ResNet50 ImageNet, fine-tuning de las últimas 40 capas y 3 clases (`benign`, `malignant`, `normal`).



In [1]:
# ============================================================
# 1. GOOGLE DRIVE + CONFIGURACIÓN
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys, json, shutil, zipfile, random

import yaml


DRIVE_ROOT = Path("/content/drive/MyDrive/VisionComputerIII/")
PROJECT_ZIP = DRIVE_ROOT / "SmartEye_project.zip"
PROJECT_ROOT = Path("/content/SmartEye")
DATA_ZIP = DRIVE_ROOT / "data.zip"
CSV_SOURCE = DRIVE_ROOT / "input_final.csv"


# Resultados persistentes en Drive
PERSIST_ROOT = DRIVE_ROOT
for p in [
    PERSIST_ROOT / "models" / "checkpoints",
    PERSIST_ROOT / "models" / "final",
    PERSIST_ROOT / "results" / "metrics",
    PERSIST_ROOT / "results" / "figures",
    PERSIST_ROOT / "results" / "confusion_matrices",
    PERSIST_ROOT / "logs",
]:
    p.mkdir(parents=True, exist_ok=True)

EXPERIMENT_NAME = "vit_no_clahe_no_ft_2"
EXPERIMENT_ROOT = PERSIST_ROOT / "results" / "experiments" / EXPERIMENT_NAME
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive:", DRIVE_ROOT)
print("Experimento:", EXPERIMENT_ROOT)


# ============================================================
# ACTUALIZAR PROYECTO DESDE EL ZIP
# ============================================================

print("\nActualizando proyecto...")

if not PROJECT_ZIP.exists():
    raise FileNotFoundError(
        f"No se encontró el ZIP: {PROJECT_ZIP}"
    )

# Eliminar proyecto anterior
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# EXTRAER SOLO EL CONTENIDO DEL PROYECTO REAL
# ============================================================

ZIP_PREFIX = "SmartEye_project/SmartEye_project/"

with zipfile.ZipFile(PROJECT_ZIP, "r") as zip_ref:

    for member in zip_ref.namelist():

        # Ignorar directorios y cualquier archivo que no
        # pertenezca al proyecto interno
        if not member.startswith(ZIP_PREFIX):
            continue

        relative_path = member[len(ZIP_PREFIX):]

        if not relative_path:
            continue

        destination = PROJECT_ROOT / relative_path

        # Directorio
        if member.endswith("/"):
            destination.mkdir(parents=True, exist_ok=True)
            continue

        # Archivo
        destination.parent.mkdir(parents=True, exist_ok=True)

        with zip_ref.open(member) as source:
            with open(destination, "wb") as target:
                target.write(source.read())

print("Proyecto actualizado:", PROJECT_ROOT)

print("src existe:",
      (PROJECT_ROOT / "src").exists())

print("config existe:",
      (PROJECT_ROOT / "configs" / "config.yaml").exists())

print("vit existe:",
      (PROJECT_ROOT / "src" / "models" / "vit.py").exists())

Mounted at /content/drive
Drive: /content/drive/MyDrive/VisionComputerIII
Experimento: /content/drive/MyDrive/VisionComputerIII/results/experiments/vit_no_clahe_no_ft_2

Actualizando proyecto...
Proyecto actualizado: /content/SmartEye
src existe: True
config existe: True
vit existe: True


In [2]:
# ============================================================
# 2. INSTALAR DEPENDENCIAS
# ============================================================
# Reiniciar el runtime NO es necesario si Colab ya tiene TensorFlow compatible.
# Si aparece un conflicto de versiones, ejecutar esta celda y reiniciar el runtime.

!pip -q install numpy==1.26.0 pandas==2.1.4 scipy==1.11.4 scikit-learn==1.4.2 opencv-python-headless==4.9.0.80 mealpy==3.0.3

import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python <3.13,>=3.9; 1.26.1 Requires-Python <3.13,>=3.9
ERROR: Could not find a version that satisfies the requirement numpy==1.26.0 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.17.4, 1.17.5, 1.18.0, 1.18.1, 1.18.2, 1.18.3, 1.18.4, 1.18.5, 1.19.0, 1.19.1, 1.19.2, 1.19.3, 1.19.4, 1.19.5, 

In [3]:
# ============================================================
# 3. PREPARAR EL PROYECTO
# ============================================================
sys.path.insert(0, str(PROJECT_ROOT))
from pathlib import Path

print("Buscando config.yaml...")

for p in Path("/content").rglob("config.yaml"):
    print(p)
print("Proyecto:", PROJECT_ROOT)
print("src existe:", (PROJECT_ROOT / "src").exists())
print("config existe:", (PROJECT_ROOT / "configs" / "config.yaml").exists())
print("vit existe:", (PROJECT_ROOT / "src" / "models" / "vit.py").exists())

print("Proyecto:", PROJECT_ROOT)
print("src existe:", (PROJECT_ROOT / "src").exists())
print("config existe:", (PROJECT_ROOT / "configs" / "config.yaml").exists())
print("vit existe:", (PROJECT_ROOT / "src" / "models" / "vit.py").exists())

Buscando config.yaml...
/content/SmartEye/configs/config.yaml
Proyecto: /content/SmartEye
src existe: True
config existe: True
vit existe: True
Proyecto: /content/SmartEye
src existe: True
config existe: True
vit existe: True


In [4]:
# ============================================================
# 4. PREPARAR DATASET EN DISCO LOCAL DE COLAB
# ============================================================
# El ZIP de imágenes se descomprime en /content para evitar entrenar directamente
# desde Drive. El CSV también se copia al árbol del proyecto local.

LOCAL_DATA_ROOT = PROJECT_ROOT / "data" / "raw"
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not DATA_ZIP.exists():
    raise FileNotFoundError(f"No encuentro el dataset: {DATA_ZIP}")

BUTDATA = LOCAL_DATA_ROOT / "BUTData"
if not BUTDATA.exists():
    print("Descomprimiendo dataset...")
    with zipfile.ZipFile(DATA_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_ROOT)

if not CSV_SOURCE.exists():
    raise FileNotFoundError(f"No encuentro el CSV: {CSV_SOURCE}")

CSV_LOCAL = LOCAL_DATA_ROOT / "input_final.csv"
shutil.copy2(CSV_SOURCE, CSV_LOCAL)

# Si el ZIP creó una carpeta /BUTData/BUTData, corregimos la referencia.
if not (BUTDATA / "train").exists() and (LOCAL_DATA_ROOT / "BUTData" / "BUTData" / "train").exists():
    BUTDATA = LOCAL_DATA_ROOT / "BUTData" / "BUTData"

for split in ["train", "val", "test"]:
    split_path = BUTDATA / split
    if not split_path.exists():
        raise FileNotFoundError(f"Falta el split {split}: {split_path}")

print("Dataset:", BUTDATA)
print("CSV:", CSV_LOCAL)

Descomprimiendo dataset...
Dataset: /content/SmartEye/data/raw/BUTData
CSV: /content/SmartEye/data/raw/input_final.csv


In [5]:
import transformers
import tensorflow as tf
import sys
import importlib.util

print("keras_hub:", importlib.util.find_spec("keras_hub"))
print("Transformers:", transformers.__version__)
print("TensorFlow:", tf.__version__)
print("Python:", sys.version)
print("TFViTModel disponible:", hasattr(transformers, "TFViTModel"))

keras_hub: ModuleSpec(name='keras_hub', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7c269e153170>, origin='/usr/local/lib/python3.13/dist-packages/keras_hub/__init__.py', submodule_search_locations=['/usr/local/lib/python3.13/dist-packages/keras_hub'])
Transformers: 5.17.0
TensorFlow: 2.20.0
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
TFViTModel disponible: False


In [6]:
# ============================================================
# 5. IMPORTS DEL PROYECTO
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.data.preprocessing import apply_clahe
from src.data.generators import image_generators
from src.models.resnet50 import build_resnet50
from src.models.vit import build_vit
from src.training.train import compile_model, train_model
from src.evaluation.evaluate import evaluate_model
from src.evaluation.plots import plot_history

from pathlib import Path

import transformers
import tensorflow as tf
import sys

print("Transformers:", transformers.__version__)
print("TensorFlow:", tf.__version__)
print("Python:", sys.version)
print("TFViTModel disponible:", hasattr(transformers, "TFViTModel"))


SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("SmartEye imports OK")

Transformers: 5.17.0
TensorFlow: 2.20.0
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
TFViTModel disponible: False
SmartEye imports OK


In [7]:
# ============================================================
# 6. CONFIGURACIÓN DEL EXPERIMENTO
# ============================================================
import yaml
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 2
NUM_CLASSES = 3
EPOCHS = 30
LEARNING_RATE = 1e-4
#trainable_layers=TRAINABLE_LAYERS,
#TRAINABLE_LAYERS = 40
USE_CLAHE = False  # coincide con la ejecución original

CLASS_NAMES = ["benign", "malignant", "normal"]


CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

TRAINABLE_LAYERS=CONFIG["model"]["trainable_layers"],
print("CONFIG cargado correctamente")
print(CONFIG["model"])
print({
    "experiment": EXPERIMENT_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "trainable_layers": TRAINABLE_LAYERS,
    "clahe": USE_CLAHE,
    "classes": CLASS_NAMES,
})

CONFIG cargado correctamente
{'name': 'vit', 'pretrained': 'google/vit-base-patch16-224', 'fine_tuning': False, 'trainable_layers': 0}
{'experiment': 'vit_no_clahe_no_ft_2', 'image_size': (224, 224), 'batch_size': 2, 'epochs': 30, 'learning_rate': 0.0001, 'trainable_layers': (0,), 'clahe': False, 'classes': ['benign', 'malignant', 'normal']}


In [8]:
# ============================================================
# 7. CARGAR DATASET
# ============================================================
train_gen, val_gen, test_gen = image_generators(
    dataset_dir=str(BUTDATA),
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    seed=SEED,
    use_clahe=USE_CLAHE,
    preprocessing_function=apply_clahe,
)

print("class_indices:", train_gen.class_indices)
print("train:", len(train_gen), "batches")
print("val:", len(val_gen), "batches")
print("test:", len(test_gen), "batches")

# Importante para evaluación reproducible
test_gen.reset()

Found 619 images belonging to 3 classes.
Found 76 images belonging to 3 classes.
Found 80 images belonging to 3 classes.
class_indices: {'benign': 0, 'malignant': 1, 'normal': 2}
train: 310 batches
val: 38 batches
test: 40 batches


In [9]:
# ============================================================
# 8. CONSTRUIR Y COMPILAR MODELO
# ============================================================
if "vit" in EXPERIMENT_NAME:
    model = build_vit(
        input_shape=(
            CONFIG["data"]["image_size"][0],
            CONFIG["data"]["image_size"][1],
            1,
        ),
        num_classes=NUM_CLASSES,
        #trainable_layers=TRAINABLE_LAYERS,
        trainable_layers=CONFIG["model"]["trainable_layers"],
        pretrained=CONFIG["model"]["pretrained"],
    )

elif "resnet" in EXPERIMENT_NAME:
    model = build_resnet50(
        input_shape=(
            CONFIG["data"]["image_size"][0],
            CONFIG["data"]["image_size"][1],
            1,
        ),
        num_classes=NUM_CLASSES,
        trainable_layers=TRAINABLE_LAYERS,
    )

else:
    raise ValueError(
        f"Modelo no soportado: {EXPERIMENT_NAME}"
    )

# ============================================================
# COMPILAR MODELO
# ============================================================

model = compile_model(
    model,
    learning_rate=LEARNING_RATE,
)

model.summary()

ViT preset: vit_base_patch16_224_imagenet


100%|██████████| 698/698 [00:00<00:00, 651kB/s]


100%|██████████| 328M/328M [00:05<00:00, 63.6MB/s]


ViT backbone: congelado


Model: "SmartEye_ViT"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ imagen (InputLayer)             │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ grayscale_to_rgb (Lambda)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vi_t_backbone (ViTBackbone)     │ (None, 197, 768)       │    85,798,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_128 (Dense)               │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classification (Dense)          │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,905,539 (327.70 MB)

 Trainable params: 106,883 (417.51 KB)

 Non-trainable params: 85,798,656 (327.30 MB)

In [10]:
# ============================================================
# 9. CALLBACKS Y RUTAS DE SALIDA
# ============================================================
checkpoint_path = PERSIST_ROOT / "models" / "checkpoints" / f"{EXPERIMENT_NAME}.keras"
final_model_path = PERSIST_ROOT / "models" / "final" / f"{EXPERIMENT_NAME}.keras"
tensorboard_dir = PERSIST_ROOT / "logs" / EXPERIMENT_NAME

callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir=str(tensorboard_dir)),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=20,
        restore_best_weights=True,
        mode="max",
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
    ),
]

print("Checkpoint:", checkpoint_path)
print("Modelo final:", final_model_path)
print("TensorBoard:", tensorboard_dir)

Checkpoint: /content/drive/MyDrive/VisionComputerIII/models/checkpoints/vit_no_clahe_no_ft_2.keras
Modelo final: /content/drive/MyDrive/VisionComputerIII/models/final/vit_no_clahe_no_ft_2.keras
TensorBoard: /content/drive/MyDrive/VisionComputerIII/logs/vit_no_clahe_no_ft_2


In [ ]:
# ============================================================
# 10. ENTRENAMIENTO
# ============================================================
history = train_model(
    model=model,
    train_data=train_gen,
    val_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# Guardar modelo final con los pesos restaurados por EarlyStopping
model.save(final_model_path)
print("Modelo guardado en:", final_model_path)

Epoch 1/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 689s 2s/step - accuracy: 0.5525 - loss: 0.9896 - val_accuracy: 0.6447 - val_loss: 0.7271
Epoch 2/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 681s 2s/step - accuracy: 0.6559 - loss: 0.7712 - val_accuracy: 0.7237 - val_loss: 0.6740
Epoch 3/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 666s 2s/step - accuracy: 0.7237 - loss: 0.6759 - val_accuracy: 0.7105 - val_loss: 0.6660
Epoch 4/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 666s 2s/step - accuracy: 0.7480 - loss: 0.5945 - val_accuracy: 0.7500 - val_loss: 0.6118
Epoch 5/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 691s 2s/step - accuracy: 0.7932 - loss: 0.5145 - val_accuracy: 0.7500 - val_loss: 0.5678
Epoch 6/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 663s 2s/step - accuracy: 0.8158 - loss: 0.4746 - val_accuracy: 0.7632 - val_loss: 0.5531
Epoch 7/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - accuracy: 0.7948 - loss: 0.4623 - val_accuracy: 0.7237 - val_loss: 0.5308
Epoch 8/30
310/310 ━━━━━━━━━━━━━━━━━━━━ 663s 2s/step - accuracy: 0.8595 - loss: 0.3877 - val_accu

In [ ]:
'''
model.load_weights(
    checkpoint_path
)

print("Pesos cargados correctamente.")


# ------------------------------------------------------------
# COMPILAR
# ------------------------------------------------------------

model = compile_model(
    model,
    learning_rate=LEARNING_RATE,
)

print("Modelo compilado correctamente.")


# ------------------------------------------------------------
# CONTINUAR ENTRENAMIENTO
# Epoch 21 ya fue alcanzado.
# Se entrenará 22 ... 30.
# ------------------------------------------------------------

print("\nContinuando entrenamiento...")
print("Desde epoch: 21")
print("Hasta epoch: 30")

history = train_model(
    model=model,
    train_data=train_gen,
    val_data=val_gen,
    epochs=9,
    callbacks=callbacks,
)
'''

In [ ]:
# ============================================================
# 11. GUARDAR HISTORY Y MÉTRICAS DEL ENTRENAMIENTO
# ============================================================
history_path = EXPERIMENT_ROOT / "history.json"
with open(history_path, "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f, indent=2)

fig_path = EXPERIMENT_ROOT / "training_history.png"
plot_history(history, output_path=str(fig_path))
plt.show()

print("History:", history_path)
print("Figura:", fig_path)

In [ ]:
# ============================================================
# 12. EVALUACIÓN EN TEST
# ============================================================
test_gen.reset()
results = evaluate_model(model, test_gen)

metrics = {
    "experiment": EXPERIMENT_NAME,
    "accuracy": float(results["accuracy"]),
}

metrics_path = EXPERIMENT_ROOT / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

report_df = pd.DataFrame(results["classification_report"]).transpose()
report_path = EXPERIMENT_ROOT / "classification_report.csv"
report_df.to_csv(report_path)

print(json.dumps(metrics, indent=2))
print(report_df)

In [ ]:
# ============================================================
# 13. MATRIZ DE CONFUSIÓN
# ============================================================
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = results["confusion_matrix"]
cm_path = EXPERIMENT_ROOT / "confusion_matrix.png"

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
).plot(ax=ax, values_format="d")
ax.set_title(f"SmartEye — {EXPERIMENT_NAME}")
fig.tight_layout()
fig.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print("Matriz guardada en:", cm_path)

In [ ]:
# ============================================================
# 14. RESUMEN DEL EXPERIMENTO
# ============================================================
summary = {
    "experiment": EXPERIMENT_NAME,
    "classes": CLASS_NAMES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_requested": EPOCHS,
    "epochs_completed": len(history.history["loss"]),
    "learning_rate": LEARNING_RATE,
    "trainable_layers": TRAINABLE_LAYERS,
    "clahe": USE_CLAHE,
    "test_accuracy": float(results["accuracy"]),
    "model_path": str(final_model_path),
    "checkpoint_path": str(checkpoint_path),
}

summary_path = EXPERIMENT_ROOT / "experiment.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nTodos los resultados quedaron persistidos en:")
print(EXPERIMENT_ROOT)

## Artefactos generados

Al terminar la corrida se obtiene, como mínimo:

```text
MyDrive/SmartEye/
├── models/
│   ├── checkpoints/
│   │   └── resnet50_finetune40.keras
│   └── final/
│       └── resnet50_finetune40.keras
├── results/
│   └── experiments/
│       └── resnet50_finetune40/
│           ├── experiment.json
│           ├── history.json
│           ├── metrics.json
│           ├── classification_report.csv
│           ├── training_history.png
│           └── confusion_matrix.png
└── logs/
    └── resnet50_finetune40/
```

